### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [15]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path

In [17]:
### Read all PDF files in the current directory



def process_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of text chunks."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}.")
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...  {pdf_file.name}")
        
        try:
            # Try using PyMuPDFLoader first
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()
        except Exception as e:
            print(f"PyMuPDFLoader failed for {pdf_file} with error: {e}. Trying PyPDFLoader...")
            try:
                # Fallback to PyPDFLoader
                loader = PyMuPDFLoader(pdf_file)
                documents = loader.load()
            except Exception as e:
                print(f"PyPDFLoader also failed for {pdf_file} with error: {e}. Skipping this file.")
                continue
        
        all_documents.extend(documents)
        print(f"loaded {len(documents)} pages.")

    
    return all_documents


# Process all PDFs in the current directory
all_pdf_documents = process_pdfs("../data")

Found 3 PDF files in ../data.
Processing ../data/pdf/iStatementWorksheet.pdf...  iStatementWorksheet.pdf
loaded 2 pages.
Processing ../data/pdf/FocusPlanWorksheet.pdf...  FocusPlanWorksheet.pdf
loaded 3 pages.
Processing ../data/pdf/ADHDWorksheet.pdf...  ADHDWorksheet.pdf
loaded 1 pages.


In [20]:
### Text splitting get into chucks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split documents {len(documents)} into {len(split_docs)} chunks.")
    
    if split_docs:
        print(f"First chunk preview: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    
    return split_docs

In [28]:
chunks=split_documents(all_pdf_documents)

chunks

Split documents 6 into 8 chunks.
First chunk preview: "I" Statements 
Worksheet
Mentalyc IncExplore Secure AI-Powered Progress Note Automation!
"I" statements are a form of assertive communication that express your thoughts,
feelings, and needs in a clea...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-07-06T21:48:31+00:00', 'title': 'Mentalyc Cheatsheets', 'moddate': '2024-07-06T21:48:30+00:00', 'keywords': 'DAGJDqOdZO0,BAFXld6g6w4', 'author': 'Eseosa Osayimwen', 'source': '../data/pdf/iStatementWorksheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-07-06T21:48:31+00:00', 'title': 'Mentalyc Cheatsheets', 'moddate': '2024-07-06T21:48:30+00:00', 'keywords': 'DAGJDqOdZO0,BAFXld6g6w4', 'author': 'Eseosa Osayimwen', 'source': '../data/pdf/iStatementWorksheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='"I" Statements \nWorksheet\nMentalyc IncExplore Secure AI-Powered Progress Note Automation!\n"I" statements are a form of assertive communication that express your thoughts,\nfeelings, and needs in a clear and respectful manner. They focus on personal\nexperiences and avoid blaming or accusing others.\nWhy Use "I" Statements?\n"I" statements help you express emotions in a direct and non-confrontational\nmanner.\nUsing "I" statements enhances mutual understanding and promotes active\nlistening.\nThey minimize defensiveness and prevent others from feeling attacked.\n"I" statements contribute to open, respectful communication, fosterin

### Embedding and VectorStoreDB

In [29]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/newapple/Documents/GitHub/rag-pipeline/server/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        
        Initialize the embedding manager with a specified model.
        
        Args: 
            model_name (str): HuggingFace model name for sentence embeddings.
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise